# Main Experiment — Main Notebook

This notebook generates 33 experiment configurations, runs them in parallel with configurable concurrency and selection, logs outputs, and analyzes how often groups reach each of the four principles vs. disagreement.

## Prerequisites

1. **Environment**: Ensure the virtual environment is activated with all dependencies installed.
2. **API Keys**: Set up `.env` with required API keys (see `README.md` for details).
3. **Configuration**: Review the execution control flags in the first code cell.

## How to Use

1. Set `create_new_configurations = True` to generate new experiment configs
2. Set `run_experiment = True` to execute experiments
3. Run all cells in order

> **Note**: Both flags default to `False` to prevent accidental execution.

In [1]:
# Flag to run the experiment and create new configurations
run_experiment = False
create_new_configurations = False

In [2]:
# Imports
import sys, os
from pathlib import Path

# Ensure repo root on sys.path (for local package imports)
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / 'main.py').exists() and (p / 'experiment_execution').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
    return here
_REPO_ROOT = _add_repo_root_to_sys_path()

import json
import random
import shutil
import yaml
from collections import Counter
import numpy as np
from scipy.stats import chi2_contingency
from experiment_execution.utils_experiment_execution.runner import (
    list_config_files,
    select_configs,
    run_configs_in_parallel,
)


In [3]:
# Configuration paths and constants
CONFIG_DIR = _REPO_ROOT / 'experiment_execution' / 'main_experiment' / 'configs'
TERMINAL_OUTPUTS_DIR   = _REPO_ROOT / 'experiment_execution' / 'main_experiment' / 'terminal_outputs'
RESULTS_DIR= _REPO_ROOT / 'experiment_execution' / 'main_experiment' / 'results'
TRANSCRIPTS_DIR = _REPO_ROOT / 'experiment_execution' / 'main_experiment' / 'transcripts'

# Placeholder model list for participant agents

MODEL_LIST = [
    "gemini-2.5-pro",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
]

# Income class probabilities (must sum to 1.0)
# Same as in Frohlich & Oppenheimer (1992) for the initial distribution
INCOME_CLASS_PROBS = {
    'high': 0.05,
    'medium_high': 0.10,
    'medium': 0.50,
    'medium_low': 0.25,
    'low': 0.10,
}

# Ensure directories exist
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
TERMINAL_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

tuple(p.relative_to(_REPO_ROOT) for p in [CONFIG_DIR, TERMINAL_OUTPUTS_DIR, RESULTS_DIR, TRANSCRIPTS_DIR])


(PosixPath('experiment_execution/main_experiment/configs'),
 PosixPath('experiment_execution/main_experiment/terminal_outputs'),
 PosixPath('experiment_execution/main_experiment/results'),
 PosixPath('experiment_execution/main_experiment/transcripts'))

## 1. Generate 33 Configurations

In [4]:
if create_new_configurations:
    def make_agents(temp: float, rng: random.Random) -> list[dict]:
        agents = []
        for i in range(0, 5):  # 5 participant agents
            agents.append({
                'name': f'Agent_{i+1}',
                'personality': 'You are an American college student',
                'model': rng.choice(MODEL_LIST),
                'temperature': float(temp),
                'memory_character_limit': 25000,
                'reasoning_enabled': True,
            })
        return agents
    
    def build_config(temp: float, seed_val: int, rng: random.Random) -> dict:
        return {
            'language': 'English',
            'seed': int(seed_val),
            'agents': make_agents(temp, rng),
            'utility_agent_model': 'gemini-2.0-flash-lite-001',
            'utility_agent_temperature': 0.0,
            'phase2_rounds': 10,
            'distribution_range_phase2': [2, 6],
            'income_class_probabilities': INCOME_CLASS_PROBS,
            'original_values_mode': {'enabled': True},
        }
    
    # Temperatures: 11 with 0, 11 with U(0,1), 11 with U(0,1.5)
    GLOBAL_SEED = 21000
    master_rng = random.Random(GLOBAL_SEED)  # deterministic config generation
    
    temps_fixed = [0.0] * 11
    temps_u01 = [master_rng.uniform(0.0, 1.0) for _ in range(11)]
    temps_u015 = [master_rng.uniform(0.0, 1.5) for _ in range(11)]
    all_temps = temps_fixed + temps_u01 + temps_u015
    
    generated_files = []
    for idx, temp in enumerate(all_temps, start=1):
        seed_val = master_rng.randint(0, 2**31 - 1)
        cfg_rng = random.Random(seed_val)  # tie agent sampling to the seed
        cfg = build_config(temp=temp, seed_val=seed_val, rng=cfg_rng)
        fname = CONFIG_DIR / f'main_experiment_condition_{idx}_config.yaml'
        with open(fname, 'w') as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
        generated_files.append(fname)
    
    len(generated_files), generated_files[0] if generated_files else None


## 2. Run Configs (Parallel + Selective)

In [5]:
if run_experiment:
    # Discover all config files
    configs = list_config_files(CONFIG_DIR)
    print(f'Found {len(configs)} configs')
    
    # Selection controls
    [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 
    21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 
    31, 32, 33]  # e.g., [1,2,3] for the first three
    SELECT_INDICES = [15,32]
    SELECT_NAMES = None    # e.g., ['condition_1', 'condition_33']
    CONCURRENCY = 2      # adjust parallel workers
    TIMEOUT_SECONDS = None # e.g., 900 for 15 minutes per run
    
    selected = select_configs(configs, include_indices=SELECT_INDICES, include_names=SELECT_NAMES)
    print(f'Selected {len(selected)} configs to run')
    
    run_results = run_configs_in_parallel(
        selected,
        concurrency=CONCURRENCY,
        logs_dir=TERMINAL_OUTPUTS_DIR,
        results_dir=RESULTS_DIR,
        timeout_sec=TIMEOUT_SECONDS,
    )
    
    # Quick summary
    ok = sum(1 for r in run_results if r.get('ok'))
    print(f'Completed: {ok}/{len(run_results)} OK')
    run_results[:3]  # show a few

## 3. Analysis — Principles vs. Disagreements

Counts how often runs ended in consensus for each principle vs. disagreement (no consensus).

In [6]:
CATEGORIES = [
    'maximizing_floor',
    'maximizing_average',
    'maximizing_average_floor_constraint',
    'maximizing_average_range_constraint',
    'disagreement',
]

def categorize_result(result_path: Path) -> str:
    try:
        with open(result_path, 'r') as f:
            data = json.load(f)
        gi = data.get('general_information', {})
        consensus = gi.get('consensus_reached', False)
        principle = gi.get('consensus_principle')
        if consensus and principle in CATEGORIES:
            return principle
        return 'disagreement'
    except Exception:
        return 'disagreement'

counts = Counter()
result_files = sorted(RESULTS_DIR.glob('*_results.json'))
for rp in result_files:
    counts[categorize_result(rp)] += 1

# Ensure all categories are present
for cat in CATEGORIES:
    counts.setdefault(cat, 0)

# Display as a simple table
print('Category | Count')
print('---------|------')
for cat in CATEGORIES:
    print(f'{cat:38} | {counts[cat]}')

counts

Category | Count
---------|------
maximizing_floor                       | 0
maximizing_average                     | 1
maximizing_average_floor_constraint    | 29
maximizing_average_range_constraint    | 0
disagreement                           | 3


Counter({'maximizing_average_floor_constraint': 29,
         'disagreement': 3,
         'maximizing_average': 1,
         'maximizing_floor': 0,
         'maximizing_average_range_constraint': 0})

## 4. Statistical Tests — Fisher–Freeman–Halton and Cramér's V

Compare aggregated Hypothesis 1 outcomes (AI) against human outcomes as a 5×2 contingency table.

- Rows (categories): the four principles + disagreement.
- Columns (groups): Human vs AI.
- Fisher–Freeman–Halton exact test via R's `fisher.test()` when available; fallback to Chi-square otherwise.
- Cramér's V with bias correction and bootstrap CI.


In [7]:

# 1) Aggregate AI outcomes across all runs into 5 categories
ai_counts = [counts.get(cat, 0) for cat in CATEGORIES]
print('AI counts by category:', dict(zip(CATEGORIES, ai_counts)))

# 2) Specify Human counts (edit to match experiment_execution/main_experiment/image copy.png)
# Defaults below use Frohlich & Oppenheimer published values as a placeholder.
HUMAN_COUNTS = {
    'maximizing_floor': 5,
    'maximizing_average': 1,
    'maximizing_average_floor_constraint': 23,
    'maximizing_average_range_constraint': 2,
    'disagreement': 7,
}
human_counts = [HUMAN_COUNTS.get(cat, 0) for cat in CATEGORIES]
print('Human counts by category:', dict(zip(CATEGORIES, human_counts)))

# 3) Build 5×2 contingency table (rows=categories, cols=[Human, AI])
contingency = np.vstack([human_counts, ai_counts]).T  # shape (5, 2)
contingency, CATEGORIES, ['Human','AI']


AI counts by category: {'maximizing_floor': 0, 'maximizing_average': 1, 'maximizing_average_floor_constraint': 29, 'maximizing_average_range_constraint': 0, 'disagreement': 3}
Human counts by category: {'maximizing_floor': 5, 'maximizing_average': 1, 'maximizing_average_floor_constraint': 23, 'maximizing_average_range_constraint': 2, 'disagreement': 7}


(array([[ 5,  0],
        [ 1,  1],
        [23, 29],
        [ 2,  0],
        [ 7,  3]]),
 ['maximizing_floor',
  'maximizing_average',
  'maximizing_average_floor_constraint',
  'maximizing_average_range_constraint',
  'disagreement'],
 ['Human', 'AI'])

In [8]:
def fisher_freeman_halton_pvalue_r(contingency: np.ndarray) -> float | None:
    """Run Fisher-Freeman-Halton test via R's fisher.test if available.
    Returns p-value or None if Rscript not found or fails.
    """
    if shutil.which('Rscript') is None:
        return None
    r_matrix = ','.join(str(int(x)) for x in contingency.flatten(order='C'))
    nrow = contingency.shape[0]
    r_code = f"""
m <- matrix(c({r_matrix}), nrow={nrow}, byrow=TRUE);
f <- tryCatch(fisher.test(m), error=function(e) NA);
if (is.list(f)) {{ cat(f$p.value) }} else {{ cat('NA') }}
"""
    import subprocess
    try:
        out = subprocess.check_output(['Rscript', '-e', r_code], stderr=subprocess.STDOUT, text=True)
        out = out.strip()
        return float(out) if out and out != 'NA' else None
    except Exception:
        return None

p_ffh = fisher_freeman_halton_pvalue_r(contingency)

print(f'Fisher–Freeman–Halton exact test p-value: {p_ffh:.6f}')


Fisher–Freeman–Halton exact test p-value: 0.033087


In [9]:
from experiment_execution.utils_experiment_execution import (
    cramers_v,
    bias_corrected_cramers_v,
    bootstrap_cramers_v,
)

# Effect size
cramers_v_value = bias_corrected_cramers_v(contingency)
print(f"Bias-corrected Cramér's V: {cramers_v_value:.6f}")

# Bootstrap confidence interval
boot_vs, ci_lo, ci_hi = bootstrap_cramers_v(
    contingency,
    n_bootstrap=2000,
    confidence_level=0.95,
    bias_corrected=True,
)
print(f"95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")


Bias-corrected Cramér's V: 0.265347
95% CI: [0.0936, 0.4892]
